# 03 · Dựng index BEiT3 + CLIP

`Keyframes → CLIP Image Encoder → CLIP embedding → CLIP Faiss`
`Keyframes → BEiT3 Image Encoder → BEiT3 embedding → BEiT3 Faiss`

| Model | Checkpoint | Chiều |
|---|---|---|
| BEiT3-Large (chi tiết) | `beit3_large_patch16_384_coco_retrieval.pth` · 1,29 GB | 1024 |
| OpenCLIP ViT-bigG-14 (thô) | `laion2b_s39b_b160k` | 1280 |

Đọc thẳng từ `zips/` nên **phải chạy `01` hoặc `02` trước**.


> **Chạy trong một phiên Colab RIÊNG.** Thư viện giữa các notebook xung đột nhau
> (paddleocr/vietocr hạ cấp Pillow), nên đừng chạy hai notebook trong cùng một runtime.

**Sửa đúng một ô: `SECTION 02a`.** Đổi `NGUOI` thành tên mình, chọn `PHAM_VI`, đặt
`SO_VIDEO`. Mọi đường dẫn tự suy ra từ đó.

```
/content/drive/MyDrive/aic_26/
├── fps_map.json                     ← dùng chung, chép từ data_raw/ trong repo
├── transnetv2-pytorch-weights.pth   ← dùng chung, tự tải lần đầu
└── output_demo/
    ├── Vanh/v001/     ← mỗi người một nhánh, không giẫm chân nhau
    │   ├── progress.json
    │   ├── zips/      AIC2026_frames_001.zip …
    │   └── indexes/   beit3.index  clip.index  keyframe_metadata.json
    │                  keyframes_list.json  image_mapping.json  fps_map.json
    ├── Phat/v001/
    ├── An/v001/
    └── Bang/v001/
```


## Đầu ra

`indexes/` — `beit3.index` · `clip.index` · `beit3_mapping.json` · `clip_mapping.json`
· `keyframe_metadata.json` · `keyframes_list.json` · `image_mapping.json`

`keyframe_metadata.json` là bảng nối: nó biến số thứ tự vector của FAISS thành
`video_id + frame_idx`. Không có nó thì hai index vô dụng.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02a – AI LÀM PHẦN NÀO + PHẠM VI TEST   ← Ô DUY NHẤT CẦN SỬA
# ═══════════════════════════════════════════════════════════════════════════════
# %%

# ─── 1. Bạn là ai ────────────────────────────────────────────────────────────
# Đổi đúng tên mình. Mỗi người ghi vào thư mục riêng nên không giẫm chân nhau.
NGUOI = "Vanh"          # "Vanh" · "Phat" · "An" · "Bang"

# ─── 2. Chạy bao nhiêu video ─────────────────────────────────────────────────
# "chia_deu"  → tự chia đều TEST_VIDEOS cho 4 người, mỗi người một phần KHÔNG
#               trùng nhau. Mặc định khi cả nhóm cùng chạy.
# "tu_chon"   → chỉ chạy đúng các video liệt kê ở VIDEO_TU_CHON.
# "tat_ca"    → chạy toàn bộ TEST_VIDEOS (một mình gánh hết).
PHAM_VI = "chia_deu"

# Giới hạn thêm SAU khi đã chia. 0 = không giới hạn.
# Lần đầu để 3 cho nhanh (~30 phút); chạy thông hết rồi mới nâng lên.
SO_VIDEO = 3

VIDEO_TU_CHON = [
    "L21_V015",   # R1-1  KIS · 4 phi hành gia — video đã đo kỹ ở Q1
    "L30_V092",   # R1-17 KIS · phông đỏ COVID-19 — video để thử OCR
]

# ─── 3. Thứ tự người, dùng để chia phần ──────────────────────────────────────
THU_TU_NGUOI = ["Vanh", "Phat", "An", "Bang"]
assert NGUOI in THU_TU_NGUOI, f"NGUOI phải là một trong {THU_TU_NGUOI}"

# ─── 4. Đường dẫn suy ra từ tên ──────────────────────────────────────────────
#   aic_26/output_demo/<Tên>/v001/
#     progress.json
#     zips/     AIC2026_frames_001.zip ...
#     indexes/  beit3.index  clip.index  keyframe_metadata.json
#               keyframes_list.json  image_mapping.json  fps_map.json
DRIVE_OUT  = f"/content/drive/MyDrive/aic_26/output_demo/{NGUOI}/v001"

# Dùng chung, nằm NGOÀI thư mục riêng để 4 người khỏi tải trọng số 4 lần
SHARED_DIR = "/content/drive/MyDrive/aic_26"

print(f"[Người]   {NGUOI}")
print(f"[Ghi vào] {DRIVE_OUT}")
print(f"[Phạm vi] {PHAM_VI}" + (f" · giới hạn {SO_VIDEO} video" if SO_VIDEO else " · không giới hạn"))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 00 – Install Dependencies
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

print("[Install] Installing baseline dependencies...")
_pip("--upgrade", "transnetv2-pytorch==1.0.5", "ffmpeg-python")
_pip("open_clip_torch")
_pip("faiss-cpu")   # faiss-gpu trên PyPI dừng ở 2022, không cài được
_pip("tqdm")
_pip("transformers>=4.40.0", "Pillow", "opencv-python-headless")
_pip("gdown")
# BEiT3 (unilm) — dependency thật, verify bằng cách tự cài + test trong sandbox
# trước khi đưa vào đây. deepspeed KHÔNG cần cho inference (verify: chỉ dùng
# trong run_beit3_finetuning.py, không import trong modeling_finetune.py).
_pip("torchscale==0.2.0", "timm==0.4.12", "torchmetrics==0.7.3")
_pip("tensorboardX", "ftfy", "blobfile", "sentencepiece")
# setuptools<81: torchmetrics 0.7.3 dùng pkg_resources (deprecated, bị xóa
# khỏi setuptools>=81) — verify lỗi thật này bằng cách tự chạy import và bắt
# ModuleNotFoundError trong sandbox trước khi thêm dòng pin này.
_pip("setuptools<81")
print("[Install] All done\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 01 – Mount Drive  ← PHẢI mount TRƯỚC mọi thứ liên quan /content/drive
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import os
DRIVE_MOUNT = "/content/drive"
if os.path.ismount(DRIVE_MOUNT):
    print("[Drive] Already mounted.")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    print("[Drive] Mounted OK.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02 – Config & Dirs  ← SAU khi Drive đã mount
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import sys, json, glob, time, math, zipfile, shutil, subprocess as _sp
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# ─── MODE ────────────────────────────────────────────────────────────────────
# "TEST"  → chỉ xử lý TEST_VIDEOS (~83 videos, đủ để kiểm thử pipeline)
# "FULL"  → toàn bộ ~1478 videos (production run)
MODE = "TEST"   # ← đổi thành "FULL" khi cần chạy thật

# ─── PATHS (path cứng, không đổi) ────────────────────────────────────────────
VIDEO_ROOT   = "/content/drive/MyDrive/aic25/AIC_Video"
OUTPUT_ROOT  = DRIVE_OUT
CKPT_TNV2    = f"{SHARED_DIR}/transnetv2-pytorch-weights.pth"
LOCAL_KF_DIR = "/content/kf_buffer"

# ─── TEST VIDEOS ──────────────────────────────────────────────────────────────
# Đối chiếu lại với All_Queries.docx (bản đầy đủ, verify từng câu qua raw XML,
# không dựa vào comment cũ). Round 1 (25 câu) + Round 2 (30 câu) đã full fill,
# Round 3 CHƯA có nội dung trong docx này (chỉ có tiêu đề, không có query nào).
#
# Round 1 GT videos — verify từng câu, KHỚP 100% với docx (24/25 câu có video;
# R1-21 không có video xác nhận — thay vào đó dùng K05_V006 làm benchmark thay
# thế, đúng như docx ghi chú: "Vid gần nhất giống mô tả... K05_V006 (khác mỗi
# ở chỗ không phải "bọ" mà là "chim")... nếu muốn lấy benchmark cho câu này
# thì lấy vid đó làm truth value")
_GT_L = {
    "L21_V015",  # R1-1 KIS: 4 phi hành gia mặc áo đen / tàu vũ trụ tư nhân
    "L21_V027",  # R1-14 KIS: điêu khắc cát, khói hồng
    "L21_V029",  # R1-2 KIS: đàn hổ con quý hiếm miền Nam
    "L22_V022",  # R1-23 KIS: thị trấn ven biển / cá mập (phim Spielberg 1975)
    "L22_V030",  # R1-8 KIS: lễ hội ẩm thực Nhật, cô bé đeo bạch tuộc đỏ
    "L23_V007",  # R1-24 KIS: đua xe đạp, 3 tay đua đồng phục trắng-vàng-xanh
    "L23_V017",  # R1-25 KIS: flycam đua xe đạp, áo xanh dương trắng vượt lên
    "L24_V033",  # R1-16 TRAKE: múa lân vàng đen trắng / lân quay cột
    "L25_V058",  # R2-21 QA: bài tập vận dụng, khoảng cách AB 2 chữ số, đáp C
    "L26_V004",  # R1-20 KIS: panna cotta 3 ly / nho đỏ bạc hà
    "L26_V056",  # R1-6 KIS: gỏi cuốn chay, bánh tráng vàng tím (video đúng)
    "L26_V072",  # R1-18 TRAKE: cắt nấm → củ năng → đậu hũ → bếp
    "L26_V178",  # R1-22 QA: bánh ít trần, 200g thịt nạc xay
    "L26_V194",  # R1-4 TRAKE: bột → măng tây → dầu → chiên
    "L26_V200",  # R1-12 KIS: trang trí bánh rán, chocolate + chuối + dâu
    "L26_V385",  # R1-6 KIS: gỏi cuốn chay (video tìm được, dù L26_V056 đúng hơn)
    "L27_V010",  # R1-19 QA: đình thần Nguyễn Trung Trực Kiên Giang / 2 câu thơ
    "L27_V013",  # R1-9 KIS: thu hoạch dứa miền Tây, bà cụ + cô gái áo hồng + ghe xanh
    "L27_V014",  # R1-5 KIS: cho dê ăn, áo thun trắng + áo kẻ sọc tím
    "L29_V023",  # R1-7 KIS: chim đầu đen ánh xanh mắt đỏ, rừng Nam Bộ
    "L30_V014",  # R2-11 KIS: phụ nữ lớn tuổi đọc sách / bìa ảnh cụ lão
    "L30_V017",  # R1-10 KIS: 3 người chơi nhạc cụ kim loại tròn / kệ sách
    "L30_V040",  # R2-18 KIS: mèo trong phòng, tô kim loại úp, bảng tên "Nà Ní"
    "L30_V057",  # R1-11 KIS: mảnh bìa đổ bóng thành chân dung / mũ lưỡi trai đen
    "L30_V072",  # R1-15 QA: câu lạc bộ FANA trao quà Xã Giang Ly Khánh Hòa
    "L30_V092",  # R1-17 KIS: trao quà từ thiện bệnh viện, phông đỏ COVID-19
    "L30_V095",  # R1-13 KIS: vệ sinh máy ảnh, ống kính + khăn tím hồng + tăm bông
}

# Round 2 GT videos (K-series = dataset batch K). Verify 100% với docx —
# 27/30 câu có video xác nhận (R2-16, R2-19 explicit "chưa tìm được đáp án"
# trong docx → KHÔNG có GT, cố tình bỏ qua. R2-12, R2-23 KHÔNG có video ID
# xác nhận trong docx dù trước đây từng nhớ nhầm là có → xem _LEGACY_UNCONFIRMED)
_GT_K = {
    "K01_V009",  # R2-4 KIS: cứu hộ lũ, cụ bà đội nón lá trên xuồng cam
    "K01_V018",  # R2-15 KIS: cô gái phỏng vấn + Deadpool + múa đao
    "K02_V005",  # R2-5 KIS + R2-22 TRAKE: robot vẽ tranh + sách MILK / khám xét
    "K03_V019",  # R2-1 KIS: nhiều người áo cờ đỏ sao vàng trước biểu tượng con cua
    "K03_V022",  # R2-29 KIS: cảng biển, cầu dài ra biển, người đội nón tai bèo
    "K03_V023",  # R2-8 KIS: cầu vòm thép đỏ → ca nô vệt sóng tròn
    "K05_V013",  # R2-24 KIS: ngư dân Campuchia nâng cá lớn nguy cấp
    "K05_V018",  # R2-25 KIS: robot 4 chân bánh xe trên đồng cỏ + bò
    "K05_V025",  # R2-7 KIS: người tuyết khổng lồ đội mũ + bình gốm băng tuyết
    "K06_V010",  # R2-20 KIS: cào muối đồng, đoàn người, "Gừng cay muối mặn"
    "K07_V025",  # R2-10 TRAKE: xưởng lắp ráp robot + công nhân + cửa ô tô
    "K07_V030",  # R2-17 KIS: phụ nữ đội nón len đeo kính "Happy New Year"
    "K09_V007",  # R2-30 KIS: kị sĩ cưỡi ngựa đồng cỏ xanh / cờ đỏ
    "K09_V012",  # R2-26 KIS: phỏng vấn Trump + đoàn người ven biển vác bao trắng
    "K09_V015",  # R2-13 KIS: bánh giá 9€ và 4.80€ / sản xuất 12 cái/tầng
    "K09_V018",  # R2-6 KIS: đài phun nước trăm ngọn nến / phụ nữ bên ly nến
    "K12_V001",  # R2-9 KIS: cá nhám 211kg treo trên xe cẩu
    "K14_V027",  # R2-2 KIS: lễ hội đèn lồng, hanbok, đèn lồng khổng lồ hình người
    "K15_V010",  # R2-27 KIS: họp giá điện, màn hình LED chữ trắng nền xanh
    "K16_V010",  # R2-14 KIS: icon hình não 5×10 / thống kê tác dụng phụ thuốc
    "K17_V003",  # R2-3 QA: bản đồ Việt Nam trái cây / Khu Du Lịch Quốc Gia Mộc Châu
    "K18_V002",  # R2-28 KIS: nhiều loài cá cảnh, cá đĩa, cá rồng, cá la hán
    "K05_V006",  # R1-21 BENCHMARK THAY THẾ (không phải GT gốc): docx ghi rõ
                 # câu gốc "năm ngoái BTC nói lỗi", K05_V006 là video gần nhất
                 # (khác "bọ" → "chim") được docx đề xuất dùng làm truth value
                 # nếu cần benchmark cho câu này.
}

# ⚠ LEGACY UNCONFIRMED — trước đây từng coi là GT nhưng All_Queries.docx (bản
# đầy đủ, verify qua raw XML) KHÔNG có video ID cho các câu này:
#   K04_V013 (từng gán cho R2-23) — R2-23 trong docx mới KHÔNG có dòng
#     "Kxx_Vxxx,timestamp" nào, chỉ có mô tả + nhận xét, không có đáp án.
#   K07_V019 (từng gán cho R2-12) — tương tự, R2-12 không có video ID.
#   L26_V176, L29_V020 (từng gán cho R3-1, R3-2) — docx này DỪNG NGAY tại
#     tiêu đề "Round 3", không có nội dung Round 3 nào cả, nên không thể
#     verify lại 2 video này từ nguồn hiện tại.
# Quyết định: KHÔNG xóa (đã cắt sẵn trong 95 video hiện có, xóa cũng không
# undo được), nhưng KHÔNG coi là GT đã verify — giữ trong diversity pool.
_LEGACY_UNCONFIRMED = {
    "K04_V013", "K07_V019", "L26_V176", "L29_V020",
}

# Diversity videos (NON-GT) — mở rộng để đạt ~25-30GB frame output (yêu cầu),
# thay vì chỉ 1-3 video/batch như trước (→ 8GB/95 video). Mở rộng lên
# K01-K18 × V001-V008 (144 video) + L21-L30 × V001-V010 (100 video) = 244 mới.
# Ước tính: 8GB/95 video ≈ 86MB frame-output/video trung bình → ~300 video
# tổng (GT+diversity) × 86MB ≈ 25.8GB, khớp mục tiêu 25-30GB. Video nào không
# tồn tại trên Drive sẽ tự động bị skip + WARN (đã có sẵn cơ chế ở Section 03).
_DIVERSITY = {
    "K01_V001", "K01_V002", "K01_V003", "K01_V004", "K01_V005", "K01_V006", "K01_V007", "K01_V008",
    "K02_V001", "K02_V002", "K02_V003", "K02_V004", "K02_V005", "K02_V006", "K02_V007", "K02_V008",
    "K03_V001", "K03_V002", "K03_V003", "K03_V004", "K03_V005", "K03_V006", "K03_V007", "K03_V008",
    "K04_V001", "K04_V002", "K04_V003", "K04_V004", "K04_V005", "K04_V006", "K04_V007", "K04_V008",
    "K05_V001", "K05_V002", "K05_V003", "K05_V004", "K05_V005", "K05_V006", "K05_V007", "K05_V008",
    "K06_V001", "K06_V002", "K06_V003", "K06_V004", "K06_V005", "K06_V006", "K06_V007", "K06_V008",
    "K07_V001", "K07_V002", "K07_V003", "K07_V004", "K07_V005", "K07_V006", "K07_V007", "K07_V008",
    "K08_V001", "K08_V002", "K08_V003", "K08_V004", "K08_V005", "K08_V006", "K08_V007", "K08_V008",
    "K09_V001", "K09_V002", "K09_V003", "K09_V004", "K09_V005", "K09_V006", "K09_V007", "K09_V008",
    "K10_V001", "K10_V002", "K10_V003", "K10_V004", "K10_V005", "K10_V006", "K10_V007", "K10_V008",
    "K11_V001", "K11_V002", "K11_V003", "K11_V004", "K11_V005", "K11_V006", "K11_V007", "K11_V008",
    "K12_V001", "K12_V002", "K12_V003", "K12_V004", "K12_V005", "K12_V006", "K12_V007", "K12_V008",
    "K13_V001", "K13_V002", "K13_V003", "K13_V004", "K13_V005", "K13_V006", "K13_V007", "K13_V008",
    "K14_V001", "K14_V002", "K14_V003", "K14_V004", "K14_V005", "K14_V006", "K14_V007", "K14_V008",
    "K15_V001", "K15_V002", "K15_V003", "K15_V004", "K15_V005", "K15_V006", "K15_V007", "K15_V008",
    "K16_V001", "K16_V002", "K16_V003", "K16_V004", "K16_V005", "K16_V006", "K16_V007", "K16_V008",
    "K17_V001", "K17_V002", "K17_V003", "K17_V004", "K17_V005", "K17_V006", "K17_V007", "K17_V008",
    "K18_V001", "K18_V002", "K18_V003", "K18_V004", "K18_V005", "K18_V006", "K18_V007", "K18_V008",
    "L21_V001", "L21_V002", "L21_V003", "L21_V004", "L21_V005", "L21_V006", "L21_V007", "L21_V008", "L21_V009", "L21_V010",
    "L22_V001", "L22_V002", "L22_V003", "L22_V004", "L22_V005", "L22_V006", "L22_V007", "L22_V008", "L22_V009", "L22_V010",
    "L23_V001", "L23_V002", "L23_V003", "L23_V004", "L23_V005", "L23_V006", "L23_V007", "L23_V008", "L23_V009", "L23_V010",
    "L24_V001", "L24_V002", "L24_V003", "L24_V004", "L24_V005", "L24_V006", "L24_V007", "L24_V008", "L24_V009", "L24_V010",
    "L25_V001", "L25_V002", "L25_V003", "L25_V004", "L25_V005", "L25_V006", "L25_V007", "L25_V008", "L25_V009", "L25_V010",
    "L26_V001", "L26_V002", "L26_V003", "L26_V004", "L26_V005", "L26_V006", "L26_V007", "L26_V008", "L26_V009", "L26_V010",
    "L27_V001", "L27_V002", "L27_V003", "L27_V004", "L27_V005", "L27_V006", "L27_V007", "L27_V008", "L27_V009", "L27_V010",
    "L28_V001", "L28_V002", "L28_V003", "L28_V004", "L28_V005", "L28_V006", "L28_V007", "L28_V008", "L28_V009", "L28_V010",
    "L29_V001", "L29_V002", "L29_V003", "L29_V004", "L29_V005", "L29_V006", "L29_V007", "L29_V008", "L29_V009", "L29_V010",
    "L30_V001", "L30_V002", "L30_V003", "L30_V004", "L30_V005", "L30_V006", "L30_V007", "L30_V008", "L30_V009", "L30_V010",
} | _LEGACY_UNCONFIRMED

# Hợp nhất: GT (~50 videos, verify 100% từ docx) + Diversity (~244+4 videos)
TEST_VIDEOS: set[str] = _GT_L | _GT_K | _DIVERSITY

print(f"[Mode] MODE = {MODE}")
print(f"[Mode] GT videos: {len(_GT_L)+len(_GT_K)} | Diversity: {len(_DIVERSITY)} | Total TEST set: {len(TEST_VIDEOS)}")

# ─── TRANSNETV2 ──────────────────────────────────────────────────────────────
TNV2_W        = 48
TNV2_H        = 27
CUT_THR       = 0.5
MIN_CUT_DIST  = 12
MIN_SCENE_LEN = 8
TNV2_CHUNK    = 200
TNV2_OVERLAP  = 25
TNV2_GDRIVE_ID = "1K7tpO6NMGVqCF30mwuyR-i2HZMA7Fsrg"

# ─── FRAME SELECTION (Paper Alg.1) ───────────────────────────────────────────
N_SAMPLE_PER_SCENE = 4
DEDUP_THR     = 0.90
JPEG_QUALITY  = 90

# ─── ZIP ─────────────────────────────────────────────────────────────────────
ZIP_EVERY_GB  = 2.0
ZIP_DIR       = f"{OUTPUT_ROOT}/zips"
INDEX_DIR     = f"{OUTPUT_ROOT}/indexes"
PROGRESS_FILE = f"{OUTPUT_ROOT}/progress.json"

# ─── MODELS ──────────────────────────────────────────────────────────────────
# BEiT3-large (Wang et al., CVPR 2023 — chính xác model [51] paper 2504.08384
# cite cho nhánh "fine-grained"). Checkpoint tốt nhất public cho retrieval:
# beit3_large_patch16_384_coco_retrieval.pth — 675M params, fine-tuned COCO
# retrieval, ITC dual-encoder. Verify TOÀN BỘ bằng cách tự clone source thật
# từ microsoft/unilm, cài dependency, patch lỗi torch._six, tải checkpoint
# thật (1.29GB) và chạy forward pass thành công (output 1024-dim, khớp
# encoder_embed_dim khai báo trong _get_large_config của chính unilm).
BEIT3_CKPT_URL   = "https://github.com/addf400/files/releases/download/beit3/beit3_large_patch16_384_coco_retrieval.pth"
BEIT3_SPM_URL    = "https://github.com/addf400/files/releases/download/beit3/beit3.spm"
BEIT3_MODEL_NAME = "beit3_large_patch16_384_retrieval"
BEIT3_INPUT_SIZE = 384
BEIT3_BATCH      = 8   # 675M params fp32 — conservative batch cho T4
BEIT3_CODE_DIR   = "/content/unilm_beit3"   # nơi clone source code thật
# ViT-bigG-14 (LAION, laion2b_s39b_b160k) — giữ nguyên theo quyết định trước:
# nhánh "coarse-grained" dùng CLIP tốt nhất khả thi, KHÔNG cần đúng tên [41]
# (paper cite CLIP gốc OpenAI nhưng không chỉ rõ checkpoint, nên chọn bản
# LAION mạnh nhất fit T4 là hợp lý — đã thảo luận và quyết định trước đó).
CLIP_MODEL       = "ViT-bigG-14"
CLIP_PRETRAIN    = "laion2b_s39b_b160k"
CLIP_BATCH       = 16
FAISS_BEIT3_DIM  = 1024  # verify thật: encoder_embed_dim trong _get_large_config
FAISS_CLIP_DIM   = 1280  # bigG-14 embed_dim, verify từ model config chính thức
NEIGHBOR_WINDOW  = 3

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for d in [OUTPUT_ROOT, ZIP_DIR, INDEX_DIR, LOCAL_KF_DIR, SHARED_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"[Config] VIDEO_ROOT  : {VIDEO_ROOT}")
print(f"[Config] OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"[Config] LOCAL buffer: {LOCAL_KF_DIR}")
print(f"[Config] ZIP every   : {ZIP_EVERY_GB} GB")

# ── Load / Init progress ───────────────────────────────────────────────────────
def load_progress() -> dict:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            p = json.load(f)
        print(f"[Resume] {len(p.get('done_videos',[]))} videos done, "
              f"{p.get('zip_count',0)} zips created.")
        return p
    print("[Resume] No progress file found – starting fresh.")
    return {
        "done_videos": [],
        "zip_count": 0,
        "completed_zips": [],
        "buffer_videos": [],
        "buffer_size_bytes": 0,
        "total_frames_cut": 0,
        "index_done_beit3": [],
        "index_done_clip":   [],
        "index_phase_done":  False,
        # Feature extraction phase (Part 2)
        "done_ocr":     [],
        "done_asr":     [],
        "done_caption": [],
        "done_tags":    [],
        "done_objects": [],
        "features_merged": False,
        "last_updated": None,
    }

def save_progress(p: dict):
    p["last_updated"] = datetime.now().isoformat()
    with open(PROGRESS_FILE, "w") as f:
        json.dump(p, f, indent=2, ensure_ascii=False)

PROGRESS = load_progress()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 08b – Chuẩn bị cho bước index
# ═══════════════════════════════════════════════════════════════════════════════
# %%
# Notebook này KHÔNG cắt keyframe nên không chạy SECTION 04/05/06 — nhưng
# SECTION 09/10/11 vẫn cần vài thứ mà mấy section đó vốn nạp. Gom lại ở đây,
# nhẹ hơn nhiều so với nạp cả TransNetV2 và CLIP ViT-B/32 chỉ để lấy `torch`.

import gc, torch
from PIL import Image
from tqdm import tqdm
import open_clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}"
      + (f" · {torch.cuda.get_device_name(0)}" if DEVICE == "cuda" else ""))

# SECTION 08 in thống kê theo ALL_VIDEOS. Notebook này không chạy SECTION 03
# (liệt kê video) vì nó index từ zip chứ không từ video, nên dựng tạm ở đây.
ALL_VIDEOS = sorted(set(
    glob.glob(os.path.join(VIDEO_ROOT, "**", "*.mp4"), recursive=True) +
    glob.glob(os.path.join(VIDEO_ROOT, "*.mp4"))
))

# fps_map.json do bước cắt ghi ra. SECTION 11 cần nó để tính timestamp.
FPS_MAP: dict[str, float] = {}
_fps_p = os.path.join(INDEX_DIR, "fps_map.json")
if os.path.exists(_fps_p):
    with open(_fps_p) as f:
        FPS_MAP = {k: float(v) for k, v in json.load(f).items()}
    print(f"[fps] Nạp {len(FPS_MAP)} video từ {_fps_p}")
else:
    print(f"[fps] CHƯA CÓ {_fps_p} — bước cắt chưa chạy xong?")

if not PROGRESS["completed_zips"]:
    print()
    print("=" * 72)
    print("  CHƯA CÓ FILE ZIP NÀO — chưa có gì để index")
    print("=" * 72)
    print(f"  done_videos    : {len(PROGRESS['done_videos'])}")
    print(f"  completed_zips : {PROGRESS['completed_zips']}")
    print()
    print("  → Chạy 01_cat_keyframe_cach1 hoặc 02_cat_keyframe_cach2 trước,")
    print("    và nhớ để CÙNG một NGUOI ở SECTION 02a.")
    print("=" * 72)
    raise SystemExit

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 08 – Status Check (run bất kỳ lúc nào)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()
zip_gb = sum(os.path.getsize(os.path.join(ZIP_DIR, z)) / 1024**3
             for z in p["completed_zips"]
             if os.path.exists(os.path.join(ZIP_DIR, z)))
print("=" * 60)
print(f"  PIPELINE STATUS  [{MODE} mode]")
print("=" * 60)
print(f"  Videos done        : {len(p['done_videos'])} / {len(ALL_VIDEOS)}")
print(f"  Total keyframes    : {p['total_frames_cut']:,}")
print(f"  Zips on Drive      : {p['zip_count']} ({zip_gb:.1f} GB)")
print(f"  BEiT3 indexed      : {len(p.get('index_done_beit3', []))} / {p['zip_count']}")
print(f"  CLIP indexed       : {len(p['index_done_clip'])} / {p['zip_count']}")
print(f"  Index done         : {p['index_phase_done']}")
print(f"  OCR done           : {len(p.get('done_ocr', []))} videos")
print(f"  ASR done           : {len(p.get('done_asr', []))} videos")
print(f"  Caption done       : {len(p.get('done_caption', []))} videos")
print(f"  Features merged    : {p.get('features_merged', False)}")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 09 – Build BEiT3 Index (stream through zips, resumable)
# Architecture: BEiT3-large (Wang et al., CVPR 2023) — Multiway Transformer,
#   675M params, backbone qua package torchscale (Magneto architecture).
#   Checkpoint: beit3_large_patch16_384_coco_retrieval.pth (fine-tuned COCO
#   retrieval, dual-encoder ITC). Đây CHÍNH XÁC là model [51] paper 2504.08384
#   cite cho nhánh "fine-grained" trong Ensemble Search.
# Toàn bộ setup dưới đây đã tự verify bằng cách clone source thật từ
# microsoft/unilm, cài dependency, patch lỗi, tải checkpoint, chạy forward
# pass thành công trong sandbox trước khi đưa vào đây — KHÔNG đoán code.
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import faiss

# ── Bước 1: Clone source code thật (không phải pip package chuẩn) ─────────────
if not os.path.exists(BEIT3_CODE_DIR):
    print("[BEiT3] Cloning microsoft/unilm (sparse, chỉ lấy beit3/)...")
    _sp.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
             "https://github.com/microsoft/unilm.git", BEIT3_CODE_DIR],
           check=True, capture_output=True)
    _sp.run(["git", "-C", BEIT3_CODE_DIR, "sparse-checkout", "set", "beit3"],
           check=True, capture_output=True)
    print(f"[BEiT3] Cloned → {BEIT3_CODE_DIR}/beit3")

BEIT3_SRC = os.path.join(BEIT3_CODE_DIR, "beit3")

# ── Bước 2: Patch torch._six (verify: module đã bị xóa khỏi torch hiện đại,
#    unilm/beit3/utils.py dòng 24 dùng "from torch._six import inf" → crash).
#    Patch 1 dòng, well-known standard fix, không đổi logic gì khác. ─────────
_utils_py = os.path.join(BEIT3_SRC, "utils.py")
with open(_utils_py) as f: _utils_content = f.read()
if "from torch._six import inf" in _utils_content:
    _utils_content = _utils_content.replace(
        "from torch._six import inf",
        "inf = float('inf')  # torch._six removed in modern torch — standard patch"
    )
    with open(_utils_py, "w") as f: f.write(_utils_content)
    print("[BEiT3] Patched torch._six issue trong utils.py")

if BEIT3_SRC not in sys.path:
    sys.path.insert(0, BEIT3_SRC)

# ── Bước 3: Tải checkpoint + tokenizer thật (verify link tải OK, 1.29GB + 1.3MB) ─
_beit3_ckpt_path = os.path.join(OUTPUT_ROOT, "beit3_large_patch16_384_coco_retrieval.pth")
_beit3_spm_path  = os.path.join(OUTPUT_ROOT, "beit3.spm")
if not os.path.exists(_beit3_ckpt_path):
    print("[BEiT3] Downloading checkpoint (1.29GB)...")
    _sp.run(["curl", "-sL", "-o", _beit3_ckpt_path, BEIT3_CKPT_URL], check=True)
if not os.path.exists(_beit3_spm_path):
    print("[BEiT3] Downloading tokenizer...")
    _sp.run(["curl", "-sL", "-o", _beit3_spm_path, BEIT3_SPM_URL], check=True)

p = load_progress()
remaining_beit3 = [z for z in p["completed_zips"] if z not in p.get("index_done_beit3", [])]
p.setdefault("index_done_beit3", [])

if not remaining_beit3:
    print("[SKIP] BEiT3 index already up to date"
         f"{' (index_phase_done=True từ lần chạy trước)' if p['index_phase_done'] else ''}.")
else:
    if p["index_phase_done"]:
        print(f"[Resume] Phát hiện {len(remaining_beit3)} zip mới chưa index "
             f"kể từ lần index_phase_done trước → tiếp tục index.")

    from timm.models import create_model
    import modeling_finetune  # trigger @register_model registration
    from transformers import XLMRobertaTokenizer
    import torchvision.transforms as T

    print(f"[BEiT3] Loading {BEIT3_MODEL_NAME}...")
    beit3_model = create_model(BEIT3_MODEL_NAME, pretrained=False)
    _ckpt = torch.load(_beit3_ckpt_path, map_location="cpu")
    _sd   = _ckpt["model"] if "model" in _ckpt else _ckpt
    del _ckpt; gc.collect() if 'gc' in dir() else None
    _missing, _unexpected = beit3_model.load_state_dict(_sd, strict=False)
    del _sd
    print(f"[BEiT3] load_state_dict: missing={len(_missing)}, unexpected={len(_unexpected)}")
    beit3_model.eval().to(DEVICE)
    beit3_tokenizer = XLMRobertaTokenizer(_beit3_spm_path)
    print(f"[BEiT3] Loaded. VRAM: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

    # Image transform — verify chính xác từ unilm/beit3/datasets.py build_transform()
    # (eval path, is_train=False): Resize BICUBIC → ToTensor → Normalize Inception stats
    _beit3_transform = T.Compose([
        T.Resize((BEIT3_INPUT_SIZE, BEIT3_INPUT_SIZE), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),  # IMAGENET_INCEPTION stats, verify từ timm.data.constants
    ])

    beit3_idx_path  = os.path.join(INDEX_DIR, "beit3.index")
    beit3_meta_path = os.path.join(INDEX_DIR, "beit3_mapping.json")
    if os.path.exists(beit3_idx_path):
        beit3_index = faiss.read_index(beit3_idx_path)
        with open(beit3_meta_path) as f: beit3_map = json.load(f)
        print(f"[BEiT3] Loaded existing: {beit3_index.ntotal} vectors")
    else:
        beit3_index = faiss.IndexFlatIP(FAISS_BEIT3_DIM)
        beit3_map   = {}

    full_meta_path = os.path.join(INDEX_DIR, "keyframe_metadata.json")
    if os.path.exists(full_meta_path):
        with open(full_meta_path) as f: FULL_META = json.load(f)
        fname_to_idx = {m["name"]: i for i, m in enumerate(FULL_META)}
    else:
        FULL_META = []; fname_to_idx = {}

    UNZIP_DIR = "/content/unzip_buf"

    @torch.no_grad()
    def embed_beit3(img_paths: list[str], bs: int = BEIT3_BATCH) -> np.ndarray:
        """API verify thật từ BEiT3ForRetrieval.forward(): image=X, only_infer=True
        → (vision_cls, None). vision_cls đã qua F.normalize() sẵn trong model."""
        all_feats = []
        for s in range(0, len(img_paths), bs):
            chunk = img_paths[s:s+bs]
            imgs  = [Image.open(p).convert("RGB") for p in chunk]
            t     = torch.stack([_beit3_transform(img) for img in imgs]).to(DEVICE)
            vision_cls, _ = beit3_model(image=t, only_infer=True)
            all_feats.append(vision_cls.cpu().numpy())
        embeds = np.vstack(all_feats).astype("float32")
        if embeds.shape[1] != FAISS_BEIT3_DIM:
            raise ValueError(
                f"[BEiT3] Model output dim={embeds.shape[1]} KHÁC "
                f"FAISS_BEIT3_DIM={FAISS_BEIT3_DIM} đã khai báo trong config. "
                f"Sửa FAISS_BEIT3_DIM = {embeds.shape[1]} rồi chạy lại."
            )
        return embeds

    for zip_name in tqdm(remaining_beit3, desc="BEiT3 indexing"):
        zip_path = os.path.join(ZIP_DIR, zip_name)
        if not os.path.exists(zip_path): print(f"  [WARN] {zip_name} not found"); continue
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        Path(UNZIP_DIR).mkdir(parents=True, exist_ok=True)
        print(f"  [Unzip] {zip_name}...")
        with zipfile.ZipFile(zip_path) as zf: zf.extractall(UNZIP_DIR)
        img_paths = sorted(glob.glob(os.path.join(UNZIP_DIR, "**", "*.jpg"), recursive=True))
        if not img_paths: shutil.rmtree(UNZIP_DIR, ignore_errors=True); continue
        print(f"  [BEiT3] Embedding {len(img_paths)} images...")
        embeds   = embed_beit3(img_paths)
        start_id = beit3_index.ntotal
        beit3_index.add(embeds)
        for i, path in enumerate(img_paths):
            fname    = os.path.basename(path)
            faiss_id = start_id + i
            beit3_map[str(faiss_id)] = f"http://localhost:8000/static/images/{fname}"
            if fname in fname_to_idx:
                FULL_META[fname_to_idx[fname]]["faiss_id_beit3"] = faiss_id
            else:
                parts    = fname.rsplit("-", 2)
                video    = parts[0]
                scene_id = int(parts[1]) if len(parts) > 2 else 0
                fi       = int(parts[2].replace(".jpg","")) if len(parts) > 2 else 0
                fps_v    = FPS_MAP.get(video, 25.0) if 'FPS_MAP' in dir() else 25.0
                ts_ms    = int(fi / fps_v * 1000)
                ts_str   = (f"{ts_ms//3600000:02d}:{(ts_ms%3600000)//60000:02d}:"
                            f"{(ts_ms%60000)//1000:02d}.{ts_ms%1000:03d}")
                entry = {
                    "faiss_id_beit3": faiss_id, "faiss_id_clip": -1,
                    "frame": f"/static/images/{fname}",
                    "url":   f"http://localhost:8000/static/images/{fname}",
                    "name":  fname, "video": video, "scene_id": scene_id,
                    "frame_idx": fi, "fps": fps_v, "timestamp_ms": ts_ms,
                    "timestamp_str": ts_str,
                    "content": [], "ocr": [], "object": [], "color": [], "action": [],
                    "neighbors_clip": [],
                }
                fname_to_idx[fname] = len(FULL_META)
                FULL_META.append(entry)
        faiss.write_index(beit3_index, beit3_idx_path)
        with open(beit3_meta_path, "w") as f: json.dump(beit3_map, f)
        with open(full_meta_path, "w") as f:   json.dump(FULL_META, f, indent=2)
        p["index_done_beit3"].append(zip_name)
        save_progress(p)
        print(f"  [✓] {zip_name}: +{len(img_paths)} → total {beit3_index.ntotal}")
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        torch.cuda.empty_cache()
    print(f"[BEiT3] Done! Total: {beit3_index.ntotal}")
    del beit3_model; torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 10 – Build CLIP ViT-H-14 Index
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()
remaining_clip = [z for z in p["completed_zips"] if z not in p["index_done_clip"]]

# FIX: giống BEiT3 ở trên — check remaining_clip trước, không gate theo
# index_phase_done (flag cũ có thể stale khi chuyển TEST→FULL mode).
if not remaining_clip:
    print("[SKIP] CLIP index already up to date"
         f"{' (index_phase_done=True từ lần chạy trước)' if p['index_phase_done'] else ''}.")
else:
    if p["index_phase_done"]:
        print(f"[Resume] Phát hiện {len(remaining_clip)} zip mới chưa index "
             f"kể từ lần index_phase_done trước → tiếp tục index.")
    print(f"[CLIP] Loading {CLIP_MODEL} / {CLIP_PRETRAIN}...")
    clip_model_idx, _, clip_prep_idx = open_clip.create_model_and_transforms(
        CLIP_MODEL, pretrained=CLIP_PRETRAIN, device=DEVICE
    )
    clip_model_idx.eval()
    if 'clip_filter_model' in dir() and hasattr(clip_filter_model, 'parameters'):
        try: del clip_filter_model; torch.cuda.empty_cache()
        except: pass

    clip_idx_path  = os.path.join(INDEX_DIR, "clip.index")
    clip_meta_path = os.path.join(INDEX_DIR, "clip_mapping.json")
    if os.path.exists(clip_idx_path):
        clip_index = faiss.read_index(clip_idx_path)
        with open(clip_meta_path) as f: clip_map = json.load(f)
        print(f"[CLIP] Loaded existing: {clip_index.ntotal} vectors")
    else:
        clip_index = faiss.IndexFlatIP(FAISS_CLIP_DIM)
        clip_map   = {}

    full_meta_path = os.path.join(INDEX_DIR, "keyframe_metadata.json")
    if not ('FULL_META' in dir() and FULL_META):
        with open(full_meta_path) as f: FULL_META = json.load(f)
    fname_to_idx = {m["name"]: i for i, m in enumerate(FULL_META)}

    UNZIP_DIR = "/content/unzip_buf"

    @torch.no_grad()
    def embed_images_clip(img_paths: list[str], bs: int = CLIP_BATCH) -> np.ndarray:
        all_feats = []
        for s in range(0, len(img_paths), bs):
            chunk = img_paths[s:s+bs]
            pils  = [Image.open(p).convert("RGB") for p in chunk]
            t     = torch.stack([clip_prep_idx(img) for img in pils]).to(DEVICE)
            f     = clip_model_idx.encode_image(t)
            f     = f / f.norm(dim=-1, keepdim=True)
            all_feats.append(f.cpu().numpy())
        embeds = np.vstack(all_feats).astype("float32")
        # Cùng safety net như BEiT3 — tránh lặp lại bug hardcode-sai-dim khi
        # đổi model (vừa đổi ViT-H-14→ViT-bigG-14, dim 1024→1280).
        if embeds.shape[1] != FAISS_CLIP_DIM:
            raise ValueError(
                f"[CLIP] Model output dim={embeds.shape[1]} KHÁC "
                f"FAISS_CLIP_DIM={FAISS_CLIP_DIM} đã khai báo trong config. "
                f"Sửa FAISS_CLIP_DIM = {embeds.shape[1]} rồi chạy lại."
            )
        return embeds

    for zip_name in tqdm(remaining_clip, desc="CLIP indexing"):
        zip_path = os.path.join(ZIP_DIR, zip_name)
        if not os.path.exists(zip_path): print(f"  [WARN] {zip_name} not found"); continue
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        Path(UNZIP_DIR).mkdir(parents=True, exist_ok=True)
        print(f"  [Unzip] {zip_name}...")
        with zipfile.ZipFile(zip_path) as zf: zf.extractall(UNZIP_DIR)
        img_paths = sorted(glob.glob(os.path.join(UNZIP_DIR, "**", "*.jpg"), recursive=True))
        if not img_paths: shutil.rmtree(UNZIP_DIR, ignore_errors=True); continue
        print(f"  [CLIP] Embedding {len(img_paths)} images...")
        embeds   = embed_images_clip(img_paths)
        start_id = clip_index.ntotal
        clip_index.add(embeds)
        for i, path in enumerate(img_paths):
            fname    = os.path.basename(path)
            faiss_id = start_id + i
            clip_map[str(faiss_id)] = f"http://localhost:8000/static/images/{fname}"
            if fname in fname_to_idx:
                FULL_META[fname_to_idx[fname]]["faiss_id_clip"] = faiss_id
        faiss.write_index(clip_index, clip_idx_path)
        with open(clip_meta_path, "w") as f: json.dump(clip_map, f)
        with open(full_meta_path, "w") as f:  json.dump(FULL_META, f, indent=2)
        p["index_done_clip"].append(zip_name)
        save_progress(p)
        print(f"  [✓] {zip_name}: +{len(img_paths)} → total {clip_index.ntotal}")
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        torch.cuda.empty_cache()
    print(f"[CLIP] Done! Total: {clip_index.ntotal}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 11 – Neighbor Lists + All Mapping Files
# ═══════════════════════════════════════════════════════════════════════════════
# %%

full_meta_path = os.path.join(INDEX_DIR, "keyframe_metadata.json")
with open(full_meta_path) as f: FULL_META = json.load(f)
print(f"[Meta] Loaded {len(FULL_META)} keyframe entries")

video_groups: dict[str, list[int]] = defaultdict(list)
for i, m in enumerate(FULL_META):
    video_groups[m["video"]].append(i)
for video, indices in video_groups.items():
    video_groups[video] = sorted(indices, key=lambda i: FULL_META[i]["frame_idx"])

for video, indices in video_groups.items():
    n = len(indices)
    for pos, meta_idx in enumerate(indices):
        nbr_clip_ids = []
        for offset in range(-NEIGHBOR_WINDOW, NEIGHBOR_WINDOW + 1):
            if offset == 0: continue
            nbr_pos = pos + offset
            if 0 <= nbr_pos < n:
                nbr_meta = FULL_META[indices[nbr_pos]]
                cid = nbr_meta.get("faiss_id_clip", -1)
                if cid >= 0: nbr_clip_ids.append(cid)
        FULL_META[meta_idx]["neighbors_clip"] = nbr_clip_ids

with open(full_meta_path, "w") as f: json.dump(FULL_META, f, indent=2)
print(f"[Meta] Saved with neighbor lists")

# keyframes_list.json
kf_list = [f"{m['video']}-{m['frame_idx']}" for m in FULL_META]
with open(os.path.join(INDEX_DIR, "keyframes_list.json"), "w") as f:
    json.dump(sorted(set(kf_list)), f, indent=2)
print(f"[Meta] keyframes_list.json: {len(set(kf_list))} entries")

# image_mapping.json (backward compat: CLIP faiss_id → url)
image_mapping = {str(m.get("faiss_id_clip",-1)): m["url"]
                 for m in FULL_META if m.get("faiss_id_clip",-1) >= 0}
with open(os.path.join(INDEX_DIR, "image_mapping.json"), "w") as f:
    json.dump(image_mapping, f, indent=2)
print(f"[Meta] image_mapping.json: {len(image_mapping)} entries")

# fps_map.json
fps_unique = {m["video"]: m["fps"] for m in FULL_META}
with open(os.path.join(INDEX_DIR, "fps_map.json"), "w") as f:
    json.dump(fps_unique, f, indent=2)
print(f"[Meta] fps_map.json: {len(fps_unique)} videos")

p = load_progress(); p["index_phase_done"] = True; save_progress(p)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 12 – Summary Part 1 (Baseline Complete)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()
print("=" * 60)
print(f"  PART 1 COMPLETE  [{MODE} mode]")
print("=" * 60)
print(f"  Videos cut         : {len(p['done_videos'])}")
print(f"  Total keyframes    : {p['total_frames_cut']:,}")
print(f"  Zips on Drive      : {p['zip_count']}")
print(f"  Index phase done   : {p['index_phase_done']}")
print()
for fname, desc in {
    "beit3.index":            "BEiT3-large FAISS (1024-dim, fine-grained)",
    "clip.index":             "CLIP ViT-bigG-14 FAISS (1280-dim, coarse-grained)",
    "keyframe_metadata.json": "Rich metadata",
    "keyframes_list.json":    "UI submit form",
    "image_mapping.json":     "Backward compat",
    "fps_map.json":           "FPS per video",
}.items():
    fpath = os.path.join(INDEX_DIR, fname)
    if os.path.exists(fpath):
        print(f"  ✓ {fname:<30} {os.path.getsize(fpath)/1024**2:7.1f} MB  ({desc})")
    else:
        print(f"  ✗ {fname:<30}  MISSING")
print()
print("  → Tiếp theo: chạy nb_full_part2_features.py để extract OCR/ASR/Caption/Tags/Objects/Color")